In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score,roc_curve, auc



### Printing hyperparameters

In [3]:
#1.Logistic Regression
#Instantiate
lr = LogisticRegression()
#Parameters
param_lr = lr.get_params()
print("Logistic Regression Hyperparameters:")
#Loop through the parameters and print them
for param, value in param_lr.items():
    print(f"{param}: {value}")
    
    

Logistic Regression Hyperparameters:
C: 1.0
class_weight: None
dual: False
fit_intercept: True
intercept_scaling: 1
l1_ratio: None
max_iter: 100
multi_class: auto
n_jobs: None
penalty: l2
random_state: None
solver: lbfgs
tol: 0.0001
verbose: 0
warm_start: False


In [4]:
#2.Decision tree classifier
dt=DecisionTreeClassifier()
param_dt=dt.get_params()
print("Decision Tree Classifier Hyperparameters")
for param, value in param_dt.items():
    print(f"{param}:{value}")


Decision Tree Classifier Hyperparameters
ccp_alpha:0.0
class_weight:None
criterion:gini
max_depth:None
max_features:None
max_leaf_nodes:None
min_impurity_decrease:0.0
min_impurity_split:None
min_samples_leaf:1
min_samples_split:2
min_weight_fraction_leaf:0.0
presort:deprecated
random_state:None
splitter:best


### Hypeparemeter tuning

In [5]:
df=pd.read_csv('churn.csv')
df.head()

,state,account length,area code,phone number,international plan,voice mail plan,number vmail messages,total day minutes,total day calls,total day charge,...,total eve calls,total eve charge,total night minutes,total night calls,total night charge,total intl minutes,total intl calls,total intl charge,customer service calls,churn
0,KS,128,415,382-4657,no,yes,25,265.1,110,45.07,...,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,371-7191,no,yes,26,161.6,123,27.47,...,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,358-1921,no,no,0,243.4,114,41.38,...,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,375-9999,yes,no,0,299.4,71,50.90,...,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,330-6626,yes,no,0,166.7,113,28.34,...,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


In [6]:
#defining the features
y=df['churn']
X=df.drop(columns=['churn','state','phone number','international plan','voice mail plan'])
#Split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [7]:
dt_c=DecisionTreeClassifier()
#defining hyperparameters and possible values
param_grid={
    'criterion':['gini','entropy'],
    'max_depth':[None,5,10,15],
    'min_samples_split':[2,4,8],
    'min_samples_leaf':[1,2,3]
}
#Create grid search
grid_search=GridSearchCV(dt_c,param_grid,cv=5,scoring='accuracy')
#Fitting the model
grid_search.fit(X_train,y_train)
#Printing hyperparameters
best_params=grid_search.best_params_
best_params

{'criterion': 'gini',
 'max_depth': 5,
 'min_samples_leaf': 1,
 'min_samples_split': 2}

### Creating a Pipeline

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
# Loading the data set
df1=pd.read_csv('churn.csv')

#defining the features
y=df1['churn']
X=df1.drop(columns=['churn','state','phone number'])

#feature Types
numeric_features=X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_features=X.select_dtypes(include=['object']).columns.tolist()

#Preprocessing
numeric_tranformer=StandardScaler()
categorical_transformer=OneHotEncoder(handle_unknown='ignore')

#Calling the preprocessor
preprocessor=ColumnTransformer(
    transformers=[
        ('num',numeric_tranformer,numeric_features),
        ('cat',categorical_transformer,categorical_features)
    ]
)

#Pipeline
pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('classifier',DecisionTreeClassifier(random_state=42))
])

#Split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

#Defining a grid
grid_1 = {
    'classifier__criterion':['gini','entropy'],
    'classifier__max_depth': [None,2,3,4,5],
    'classifier__min_samples_split': [2,4,6,8],
    'classifier__min_samples_leaf': [1,2,3,4]
}

grid_search_1=GridSearchCV(pipeline,grid_1,cv=5,scoring='accuracy')
grid_search_1.fit(X_train,y_train)

#Results
print('best params:',grid_search_1.best_params_)
print('best_score:',grid_search_1.best_score_)



best params: {'classifier__criterion': 'gini', 'classifier__max_depth': 5, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 4}
best_score: 0.9396104306764762


In [ ]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3333 entries, 0 to 3332
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   account length          3333 non-null   int64  
 1   area code               3333 non-null   int64  
 2   international plan      3333 non-null   object 
 3   voice mail plan         3333 non-null   object 
 4   number vmail messages   3333 non-null   int64  
 5   total day minutes       3333 non-null   float64
 6   total day calls         3333 non-null   int64  
 7   total day charge        3333 non-null   float64
 8   total eve minutes       3333 non-null   float64
 9   total eve calls         3333 non-null   int64  
 10  total eve charge        3333 non-null   float64
 11  total night minutes     3333 non-null   float64
 12  total night calls       3333 non-null   int64  
 13  total night charge      3333 non-null   float64
 14  total intl minutes      3333 non-null   

In [ ]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3333 entries, 0 to 3332
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   state                   3333 non-null   object 
 1   account length          3333 non-null   int64  
 2   area code               3333 non-null   int64  
 3   phone number            3333 non-null   object 
 4   international plan      3333 non-null   object 
 5   voice mail plan         3333 non-null   object 
 6   number vmail messages   3333 non-null   int64  
 7   total day minutes       3333 non-null   float64
 8   total day calls         3333 non-null   int64  
 9   total day charge        3333 non-null   float64
 10  total eve minutes       3333 non-null   float64
 11  total eve calls         3333 non-null   int64  
 12  total eve charge        3333 non-null   float64
 13  total night minutes     3333 non-null   float64
 14  total night calls       3333 non-null   